# QLoRA Fine-Tuning; LLaMA 3.1 8B

In [ ]:
import sys
print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q --upgrade transformers==4.48.3 accelerate==1.3.0 datasets==3.2.0
!pip install -q --upgrade peft==0.14.0 trl==0.14.0 bitsandbytes==0.46.0
!pip install -q --upgrade matplotlib scipy scikit-learn
!pip install -q --upgrade "huggingface_hub<1.0,>=0.24.0"
!pip install -q --upgrade bitsandbytes

## Environment Setup

In [ ]:
import os
import re
import math
import torch
import torch.nn.functional as torch_F
import matplotlib.pyplot as plot
from tqdm import tqdm
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from datasets import load_dataset
from peft import PeftModel

%matplotlib inline

In [ ]:
LLM_BASE = "meta-llama/Meta-Llama-3.1-8B"
HF_DATASET = "ed-donner/pricer-data"
ADAPTER_REPO = "ed-donner/pricer-2024-09-13_13.04.39"
ADAPTER_REV = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"

TOP_K_TOKENS = 3
EVAL_N = 250

ANSI_GREEN = "\033[92m"
ANSI_YELLOW = "\033[93m"
ANSI_RED = "\033[91m"
ANSI_RESET = "\033[0m"
ANSI_BY_ZONE = {"red": ANSI_RED, "orange": ANSI_YELLOW, "green": ANSI_GREEN}

In [ ]:
from google.colab import userdata
hf_token_value = userdata.get('HF_TOKEN')

login(hf_token_value, add_to_git_credential=True)
print("Successfully authenticated with HuggingFace")

## Load Data

In [ ]:
print(f"Loading dataset: {HF_DATASET}")
data_bundle = load_dataset(HF_DATASET)
train_split = data_bundle['train']
test_split = data_bundle['test']

print(f"\nDataset loaded successfully:")
print(f"  Training examples: {len(train_split):,}")
print(f"  Test examples: {len(test_split):,}")

In [ ]:
print("Sample test example:\n")
first_example = test_split[0]
print(f"Text: {first_example['text'][:200]}...")
print(f"\nGround truth price: ${first_example['price']:.2f}")

## Quantization & Model Loading

(4-bit quantization reduces LLaMA 3.1 8B from ~32GB to ~5-6GB VRAM)

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

print("Using 4-bit NF4 quantization")

In [ ]:
print(f"Loading base model: {LLM_BASE}")

tok = AutoTokenizer.from_pretrained(LLM_BASE, trust_remote_code=True)
tok.pad_token = tok.eos_token
tok.padding_side = "right"

core_model = AutoModelForCausalLM.from_pretrained(
    LLM_BASE,
    quantization_config=quantization,
    device_map="auto",
)
core_model.generation_config.pad_token_id = tok.pad_token_id

print(f"Base model loaded - Memory: {core_model.get_memory_footprint() / 1e9:.2f} GB")

## Load PEFT Adapters

In [ ]:
print(f"Loading fine-tuned adapters: {ADAPTER_REPO}")
print(f"Revision: {ADAPTER_REV}")

qlora_model = PeftModel.from_pretrained(core_model, ADAPTER_REPO, revision=ADAPTER_REV)

print(f"Fine-tuned model ready - Total memory: {qlora_model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
def parse_price_from_text(text: str) -> float:
    # Logic equivalent to the original: pull the first number after the marker.
    marker = "Price is $"
    if marker not in text:
        return 0.0
    tail = text.split(marker, 1)[1]
    tail = tail.replace(",", "").replace("$", "")
    m = re.search(r"[-+]?\d*\.?\d+", tail)
    return float(m.group()) if m else 0.0

In [ ]:
examples = [
    "Price is $24.99",
    "Price is $1,234.50",
    "Price is $a fabulous 899.99 or so",
]

for s in examples:
    val = parse_price_from_text(s)
    print(f"{s} -> ${val:.2f}")

## Prediction Function

Top-K weighted averaging computes probability-weighted average of top K tokens.

In [ ]:
def topk_weighted_price(prompt: str, top_k: int = TOP_K_TOKENS, seed: int = 42) -> float:
    # Slight refactor: keep everything on the model device and only move what we need.
    set_seed(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_ids = tok.encode(prompt, return_tensors="pt").to(device)
    attn = torch.ones_like(input_ids, device=device)

    with torch.no_grad():
        out = qlora_model(input_ids, attention_mask=attn)
        logits = out.logits[:, -1, :]

    probs = torch_F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    prices = []
    weights = []

    for i in range(top_k):
        token_str = tok.decode(top_ids[0, i])
        p = top_probs[0, i].item()
        try:
            num = float(token_str)
        except ValueError:
            continue
        if num > 0:
            prices.append(num)
            weights.append(p)

    if not prices:
        return 0.0

    weight_sum = sum(weights)
    return sum(v * w / weight_sum for v, w in zip(prices, weights))

## Evaluation Framework

Metrics:
- Dollar Error: |prediction - truth|
- RMSLE: Root Mean Squared Log Error (penalizes relative errors)
- Hit Rate: Percentage in green zone (error < $40 OR < 20% of true price)

In [ ]:
class Scorecard:

    def __init__(self, predictor, data, title=None, size=EVAL_N):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = min(size, len(data))
        self.preds = []
        self.labels = []
        self.abs_err = []
        self.sle = []
        self.zones = []

    def _zone(self, err: float, truth: float) -> str:
        if err < 40 or err / truth < 0.2:
            return "green"
        if err < 80 or err / truth < 0.4:
            return "orange"
        return "red"

    def _eval_one(self, i: int):
        row = self.data[i]
        pred = self.predictor(row["text"])
        truth = row["price"]
        err = abs(pred - truth)

        log_err = math.log(truth + 1) - math.log(pred + 1)
        sle = log_err ** 2

        zone = self._zone(err, truth)
        short_title = row["text"].split("\n\n")[1][:30] + "..."

        self.preds.append(pred)
        self.labels.append(truth)
        self.abs_err.append(err)
        self.sle.append(sle)
        self.zones.append(zone)

        print(
            f"{ANSI_BY_ZONE[zone]}{i+1}: Guess: ${pred:,.2f} | Truth: ${truth:,.2f} | "
            f"Error: ${err:,.2f} | SLE: {sle:,.3f} | {short_title}{ANSI_RESET}"
        )

    def _plot(self, title: str):
        plot.figure(figsize=(14, 10))
        mx = max(max(self.labels), max(self.preds))

        plot.plot([0, mx], [0, mx], color='deepskyblue', lw=3, alpha=0.7, label='Perfect prediction')
        plot.scatter(self.labels, self.preds, s=20, c=self.zones, alpha=0.6)

        plot.xlabel('Ground Truth Price ($)', fontsize=12)
        plot.ylabel('Model Prediction ($)', fontsize=12)
        plot.xlim(0, mx)
        plot.ylim(0, mx)
        plot.title(title, fontsize=14, fontweight='bold')
        plot.grid(alpha=0.3)
        plot.legend()
        plot.tight_layout()
        plot.show()

    def report(self):
        avg_err = sum(self.abs_err) / self.size
        rmsle = math.sqrt(sum(self.sle) / self.size)
        hits = sum(1 for z in self.zones if z == "green")
        hit_rate = hits / self.size * 100

        title = f"{self.title} | Avg Error: ${avg_err:,.2f} | RMSLE: {rmsle:.3f} | Hit Rate: {hit_rate:.1f}%"

        print(f"\n{'='*80}")
        print("EVALUATION SUMMARY")
        print(f"{'='*80}")
        print(f"Model: {self.title}")
        print(f"Test Size: {self.size}")
        print(f"Average Dollar Error: ${avg_err:,.2f}")
        print(f"RMSLE: {rmsle:.4f}")
        print(f"Hit Rate (Green): {hit_rate:.2f}% ({hits}/{self.size})")
        print(f"{'='*80}\n")

        self._plot(title)

    def run(self):
        print(f"Running evaluation on {self.size} examples...\n")
        for i in tqdm(range(self.size), desc="Evaluating"):
            self._eval_one(i)
        self.report()

    @classmethod
    def evaluate(cls, fn, data, **kwargs):
        cls(fn, data, **kwargs).run()

In [ ]:
print(f"Loading dataset: {HF_DATASET}")
data_bundle = load_dataset(HF_DATASET)
train_split = data_bundle['train']
test_split = data_bundle['test']

print(f"\nDataset loaded successfully:")
print(f"  Training examples: {len(train_split):,}")
print(f"  Test examples: {len(test_split):,}")

## Run Evaluation

In [ ]:
Scorecard.evaluate(topk_weighted_price, test_split, title="LLaMA 3.1 8B QLoRA (400K)")